## Polygonization Exercise

In [1]:
from pygel3d import hmesh, jupyter_display as jd
jd.set_export_mode(True)
import numpy as np
from numpy.linalg import norm
from math import exp,floor
array = np.array
        
cube_faces = np.array([[[0,-0.5,-0.5],[0,0.5,-0.5],[0,0.5,0.5],[0,-0.5,0.5]],
    [[0, 0.5,-0.5],[0,-0.5,-0.5],[0,-0.5,0.5],[0,0.5,0.5]],
    [[ 0.5,0, -0.5],[-0.5,0, -0.5],[-0.5,0, 0.5],[0.5,0, 0.5]],
    [[-0.5,0, -0.5],[0.5,0, -0.5],[0.5,0, 0.5],[-0.5,0, 0.5]],
    [[-0.5,-0.5,0],[0.5,-0.5,0],[0.5,0.5,0],[-0.5,0.5,0]],
    [[ 0.5,-0.5,0],[-0.5,-0.5,0],[-0.5,0.5,0],[0.5,0.5,0]]])

neighbours = np.array([[1.,0.,0.],[-1.,0.,0.],[0.,1.,0.],[0.,-1.,0.],[0.,0.,1.],[0.,0.,-1.]])

In [2]:
class SphereVolume:
    def __init__(self,_center,_radius):
        self.center = _center
        self.radius = _radius
    def eval(self,p):
        return norm(p-self.center) - self.radius
    def grad(self,p):
        v = p - self.center
        return v / norm(v)

### The XForm Class

The class below is used to transform between the 3D grid and the domain in which your output should live. For instance, say you have a unit sphere represented as a distance field. Perhaps you want the unit sphere to exist in the domain [-1,1]^3 but use a grid of 16x16x16 polygonization cells. In this case the point [0,0,0] in the voxel grid should map to [-1,-1,-1] in the domain and [16,16,16] should map to [1,1,1]. The `XForm` class takes care of this mapping and the inverse.


In [6]:
class XForm:
    def __init__(self,Plow,Phigh,dims):
        self.dims = np.array(dims)
        self.Plow = np.array(Plow)
        self.Phigh = np.array(Phigh)
        self.scaling = (self.Phigh - self.Plow) / self.dims
    def map(self,pi):
        return self.scaling * np.array(pi) + self.Plow

### Part 1: Finish the polygonizer
You need to find the cube faces that should be generated and also project vertices back onto the isosurface

In [81]:
def smooth(m):
    # You now the drill - insert code for a gentle Laplacian smoothing
    hmesh.laplacian_smooth(m, w = 0.5, no_iters = 10)

def polygonize_volume(vol,xform,tau):
    m = hmesh.Manifold() # initialize an empty mesh
    for pi in np.ndindex(tuple(xform.dims)): # looping over all the grid points
        if vol.eval(xform.map(pi)) < tau: # check if the grid point is inside the volume (i.e. below the iso-value [tau])
            # Insert code here: For each of the six cube faces, check if neighbor is on the
            # other side of isovalue and generate face if so.
            for i in range(6): # iterating over the six faces of the cube
                neighbor_idx = pi + neighbours[i] # calculate the index of the neighboring grid point

                in_bounds = (0 <= neighbor_idx).all() and (neighbor_idx < xform.dims).all()
                if in_bounds:
                    is_outside = vol.eval(xform.map(neighbor_idx)) >= tau
                else:
                    is_outside = False

                if is_outside:
                    face_vertices = [xform.map(pi + 0.5 * neighbours[i] + vertex) for vertex in cube_faces[i]]
                    m.add_face(face_vertices) # add the face to the mesh
                        
    print("Faces added initially:", len(m.faces()))
    hmesh.stitch(m)
    print("Faces after stitch:", len(m.faces()))
    m.cleanup()
    print("Faces after cleanup:", len(m.faces()))
    smooth(m)
    pos = m.positions()
    
    # visualise the mesh at this stage
    jd.display(m)

    for v in m.vertices():
        p = pos[v]
        f_p = vol.eval(p)
        gf_p = vol.grad(p)
        g_fp_sq = np.dot(gf_p,gf_p)

        # if g_fp_sq > 1e-18: # avoid division by zero
        pos[v] = p - f_p * gf_p/g_fp_sq # project the vertex onto the isosurface


    return m

In [87]:
# the sphere radius needed to be lowered to 1.3 with resolution 16, given that 1.4 was outside the grid.
# similarly, the resolution can be increased to 32 with radius 1.4, which is still within the grid.

vol = SphereVolume([0.,0.5,0.],1.4)
xform = XForm([-1.5,-1.5,-1.5],[1.5,1.5,1.5],[32, 32, 32])
m = polygonize_volume(vol, xform, 0)
jd.display(m)

Faces added initially: 3489
Faces after stitch: 3489
Faces after cleanup: 3489


### Part 2: Create Gaussian and Gradient of Gaussian functions

In [111]:
### Exercise: Create functions that return value of 3D Gaussian and gradient. 
ssq = 0.1**2
def Gaussian(p0,p):
    # Compute and return gaussian
    d = p - p0
    return np.exp(-np.sum(d**2) / (2 * ssq))

def GaussianGrad(p0,p):
    # Compute and return gradient of gaussian
    d = p - p0
    val = Gaussian(p0, p)
    return -(d / ssq) * val


In [112]:
class Blobs:
    def __init__(self,centers):
        self.centers = centers
    def eval(self,pp):
        p = np.divmod(pp,1)[1]
        return sum([Gaussian(c, p) for c in self.centers])
    def grad(self,pp):
        p = np.divmod(pp,1)[1]
        return sum([GaussianGrad(c, p) for c in self.centers])

### Part 3: play with Blobs
Finally, when you see the blobby cube, try tweaking it to model something different.

In [123]:
cntrs = [array([0.4,0.4,0.4]) + array([0.2*p[0],0.2*p[1],0.2*p[2]]) for p in np.ndindex((2,2,2))] # rounded cube
cntrs_1 = [array([0.2,0.2,0.2]) + array([0.4*p[0],0.4*p[1],0.4*p[2]]) for p in np.ndindex((2, 2, 2))] # 8 spheres in a cube
cntrs_2 = [array([0.2, 0.2, 0.2]) + array([0.3*p[0],0.4*p[1],0.3*p[2]]) for p in np.ndindex((2, 2, 2))] # 8 spheres in a cube
vol = Blobs(cntrs_2)
xform = XForm([0,0,0],[1,1,1],[32,32,32])
m = polygonize_volume(vol, xform, 0.35)
hmesh.flip_orientation(m)
jd.display(m)

Faces added initially: 2948
Faces after stitch: 2948
Faces after cleanup: 2948


## Questions

- What is a distance field and how does it relate to a scalar field?\
A distance field is a type of scalar field. The distance field maps the distance from a given point to the nearest point on a target, often a surface. The distance can be both signed and unsigned, meanining that depending on wheter we are inside or outside the e.g. boundary, it may or may not have different signs.
- Explain the difference between primal and dual isocontouring methods.\
The difference between the methods mainly comes down to how geometric elements of the spatial grid map to the reconstructed vertices and polygons. One example may be the mesh vertices. In primal methods such as Marching-Cubes, these are placed on grid edges, while in e.g. Dual-Contouring, then are placed INSIDE the grid cells.
- Explain the method for projecting points onto the isocontour.
